# Using the `LShaped` module

`lshaped_general.ipynb` develops the L-shaped method step by step. `LShaped.jl` packages the result
as a module, and this notebook shows how to **use** it, one feature at a time:

1. a problem given as data, solved by the single-cut and the multicut methods;
2. reading the result, and the log;
3. feasibility cuts;
4. equality rows;
5. bounded recourse variables, and why the cuts are built as supporting hyperplanes;
6. one solver for the master, another for the recourse problems;
7. integer first-stage variables;
8. a problem written as JuMP models of your own, when `W` or `q` depend on the scenario;
9. sampled scenarios, reproducibly;
10. the value of perfect information and of the stochastic solution;
11. the variants of deck 04: clusters of scenarios, regularization, a starting point, and stopping
    as soon as the bounds on the EVPI or the VSS answer the question;
12. instances read from SMPS files, the standard format of stochastic programming test sets;
13. integer recourse, by the integer L-shaped method.

Every result is checked against the extensive form. The module needs JuMP; we use HiGHS, GLPK and Ipopt
as solvers and Distributions for the sampled scenarios.

In [1]:
include("LShaped.jl")
using .LShaped
using JuMP, HiGHS, GLPK, Ipopt
using LinearAlgebra, Statistics, Distributions, Printf

## 1. A problem given as data

We take the ice-cream capacity problem of `two_stages.ipynb`: four plants whose capacity $x_i$ is
bought at cost $c_i$, then three flavors produced at cost $a_{ij}$ per unit, once the demand for
flavor 1, $\xi \in \{3, 5, 7\}$ with probabilities $0.3, 0.4, 0.3$, is known:

\begin{align*}
\min_{x \geq 0}\ & c^Tx + \mathbb{E}_\xi[Q(x,\xi)]
  \quad\text{s.t.}\quad \textstyle\sum_i x_i \geq 12,\quad c^Tx \leq 120,\\
Q(x,\xi) = \min_{y \geq 0}\ & \textstyle\sum_{i,j} a_{ij}y_{ij}
  \quad\text{s.t.}\quad \textstyle\sum_j y_{ij} \leq x_i\ \forall i,
  \quad \textstyle\sum_i y_{ij} \geq d_j(\xi)\ \forall j.
\end{align*}

`TwoStageProblem` takes the data of

$$
\min_x\ c^Tx + \mathbb{E}_\xi[Q(x,\xi)] \ \text{ s.t. } Ax \gtreqless b,
\qquad
Q(x,\xi) = \min_y\ q(\xi)^Ty \ \text{ s.t. } Wy \gtreqless h(\xi) - T(\xi)x,
$$

where **each row carries its own sense**: there is no need to rewrite the problem with $\geq$ rows
only, as `two_stages.ipynb` does. Here the capacity rows are $\leq$, the demand rows $\geq$, and
the first stage has one row of each kind. `T`, `h` and `q` may be functions of the scenario or
constants; the variables are non-negative unless bounds `lb1`, `ub1` (first stage) or `lb`, `ub`
(second stage) say otherwise.

In [2]:
const NPLANTS, NFLAVORS = 4, 3
opening = [10.0, 7.0, 16.0, 6.0]
production = [40.0 24.0 4.0; 45.0 27.0 4.5; 32.0 19.2 3.2; 55.0 33.0 5.5]

# the ice-cream problem, its recourse variables stored plant by plant: yᵢⱼ is y[3(i-1)+j]
function icecream(; min_capacity = 12.0, budget = 120.0, ub = Inf, intercept = :tight)
    idx(i, j) = NFLAVORS * (i - 1) + j
    W = zeros(NPLANTS + NFLAVORS, NPLANTS * NFLAVORS)
    T = zeros(NPLANTS + NFLAVORS, NPLANTS)
    for i in 1:NPLANTS, j in 1:NFLAVORS
        W[i, idx(i, j)] = 1.0              # capacity:  Σⱼ yᵢⱼ ≤ xᵢ, i.e. Σⱼ yᵢⱼ ≤ 0 - (-1)xᵢ
        W[NPLANTS + j, idx(i, j)] = 1.0    # demand:    Σᵢ yᵢⱼ ≥ dⱼ(ξ)
    end
    for i in 1:NPLANTS
        T[i, i] = -1.0
    end
    return TwoStageProblem(
        c = opening,
        A = [ones(1, NPLANTS); opening'], senses1 = ['>', '<'], b = [min_capacity, budget],
        q = vec(permutedims(production)),
        W = W, senses2 = vcat(fill('<', NPLANTS), fill('>', NFLAVORS)),
        T = T, h = ξ -> vcat(zeros(NPLANTS), ξ, [3.0, 2.0]),
        ξ = [3.0, 5.0, 7.0], p = [0.3, 0.4, 0.3],
        ub = ub, intercept = intercept)
end

pb = icecream();

The extensive form gives the reference solution. `extensive_form` returns the JuMP model, the
first-stage solution and the optimal value.

In [3]:
_, x_ef, obj_ef = extensive_form(pb; optimizer = HiGHS.Optimizer)
@printf("optimal value %.6f at x = %s\n", obj_ef, round.(x_ef, digits = 4))

optimal value 381.853333 at x = [2.6667, 4.0, 3.3333, 2.0]


`lshaped` solves the same problem by decomposition. With `cuts = :single` (the default) it adds one
optimality cut per iteration, built from the average of the scenario subgradients:

In [4]:
single = lshaped(pb; optimizer = HiGHS.Optimizer);

Single-cut L-shaped method
  master: HiGHS | recourse: HiGHS | scenarios: 3

 iter      lower bound   upper bound          gap              θ           Q(x)
    1             -Inf   457.000000          Inf           -Inf     385.000000
    2       325.000000   400.000000           75     205.000000     280.000000
    3       364.587629   397.787629         33.2     282.319588     315.519588
    4       374.828179   388.499725        13.67     254.828179     268.499725
    5       377.369054   383.134006        5.765     257.369054     263.134006
    6       380.281437   383.134006        2.853     260.281437     264.426347
    7       380.594325   382.499270        1.905     260.594325     262.499270
    8       381.463500   382.060820       0.5973     261.463500     262.060820
    9       381.588596   382.060820       0.4722     261.588596     262.338596
   10       381.853333   381.853333   -2.842e-13     261.853333     261.853333
converged in 10 iteration(s): 9 optimality cut(s), 0 

With `cuts = :multi` it keeps one variable $\theta_s$ per scenario and adds one cut per scenario
and per iteration — the multicut version. The master problem grows faster, but it holds a better
approximation of $\mathcal{Q}$, and usually needs fewer iterations:

In [5]:
multi = lshaped(pb; optimizer = HiGHS.Optimizer, cuts = :multi);

Multi-cut L-shaped method
  master: HiGHS | recourse: HiGHS | scenarios: 3

 iter      lower bound   upper bound          gap         min θₖ           Q(x)
    1             -Inf   457.000000          Inf           -Inf     385.000000
    2       325.000000   400.000000           75      95.000000     280.000000
    3       367.500000   391.000000         23.5     184.000000     289.000000
    4       378.555556   383.302222        4.747     167.555556     263.302222
    5       380.657971   382.154348        1.496     175.130435     262.154348
    6       381.853333   381.853333    5.684e-14     175.400000     261.853333
converged in 6 iteration(s): 14 optimality cut(s), 0 feasibility cut(s)


## 2. Reading the result

The log above prints, at each iteration, the lower bound (the value of the master problem), the
upper bound, their gap, the value of $\theta$ and the expected recourse $\mathcal{Q}(x^k)$. The
lower bound is $-\infty$ until the first optimality cut, since $\theta$ is free until then.

The upper bound is that of the **incumbent**, the best first-stage solution found so far:
$c^Tx^k + \mathcal{Q}(x^k)$ need not decrease from one iteration to the next. `lshaped` returns
a named tuple:

In [6]:
keys(single)

(:x, :objective, :lower_bound, :gap, :converged, :stopped, :iterations, :optimality_cuts, :integer_cuts, :feasibility_cuts, :dropped_cuts, :clusters, :regularization, :problem, :master, :recourse, :history)

In [7]:
@printf("converged: %s in %d iterations, %d optimality cut(s), %d feasibility cut(s)\n",
        single.converged, single.iterations, single.optimality_cuts, single.feasibility_cuts)
@printf("objective %.6f, lower bound %.6f, gap %.2e\n", single.objective, single.lower_bound,
        single.gap)
println("x = ", round.(single.x, digits = 4))
println("same solution as the extensive form: ", isapprox(single.x, x_ef; atol = 1e-6))

converged: true in 10 iterations, 9 optimality cut(s), 0 feasibility cut(s)
objective 381.853333, lower bound 381.853333, gap -2.84e-13
x = [2.6667, 4.0, 3.3333, 2.0]
same solution as the extensive form: true


The decisions themselves are read with `first_stage_decision(res)` and
`second_stage_decision(res, s)`, and displayed with `print_first_stage(res)` and
`print_second_stage(res, s)`. A second-stage decision is specific to a scenario: it is obtained by
solving the recourse problem of scenario `s` at the first-stage decision returned. (The recourse
models in `res.recourse` were last solved at the last iterate, which need not be that decision.)

In [8]:
print_first_stage(single)
print_second_stage(single, 3)

First-stage decision, objective 381.853333
  x[1] = 2.666667
  x[2] = 4.0
  x[3] = 3.333333
  x[4] = 2.0
Second-stage decision, scenario 3 of 3: ξ = 7.0, probability 0.3, Q(x, ξ) = 350.333333
  y[1]  = 2.666667
  y[2]  = 0.0
  y[3]  = 0.0
  y[4]  = 1.0
  y[5]  = 3.0
  y[6]  = 0.0
  y[7]  = 3.333333
  y[8]  = 0.0
  y[9]  = 0.0
  y[10] = 0.0
  y[11] = 0.0
  y[12] = 2.0


`second_stage_decision` returns the values as a vector, in the order of the recourse variables. Here
$y_{ij}$ is stored in position $3(i-1)+j$, so the production plan of scenario 1, plant by plant and
flavor by flavor, is:

In [9]:
plan = permutedims(reshape(second_stage_decision(single, 1), NFLAVORS, NPLANTS))   # plants × flavors
round.(plan, digits = 4) .+ 0.0   # .+ 0.0 turns -0.0 into 0.0

4×3 Matrix{Float64}:
 0.0  2.6667  0.0
 0.0  0.0     2.0
 3.0  0.3333  0.0
 0.0  0.0     0.0

`history` holds one entry per iteration that produced an optimality cut: the bounds, the `value`
$c^Tx^k + \mathcal{Q}(x^k)$ of the iterate and the iterate itself. In this run the value of the
iterate goes up at some iterations while the incumbent never does:

In [10]:
println(" iter        value  upper bound  lower bound")
for h in single.history
    @printf("%5d %12.4f %12.4f %12s\n", h.iteration, h.value, h.upper_bound,
            isfinite(h.lower_bound) ? @sprintf("%.4f", h.lower_bound) : "-Inf")
end

 iter        value  upper bound  lower bound
    1     457.0000     457.0000         -Inf
    2     400.0000     400.0000     325.0000
    3     397.7876     397.7876     364.5876
    4     388.4997     388.4997     374.8282
    5     383.1340     383.1340     377.3691
    6     384.4263     383.1340     380.2814
    7     382.4993     382.4993     380.5943
    8     382.0608     382.0608     381.4635
    9     382.3386     382.0608     381.5886
   10     381.8533     381.8533     381.8533


The models are returned as well — `master` and the `recourse` model of each scenario — so that
anything JuMP knows about them can be queried, e.g. the cuts the master problem ended up with:

In [11]:
rows = all_constraints(single.master.model; include_variable_in_set_constraints = false)
cuts = filter(con -> occursin("θ", string(con)), rows)
println(length(cuts), " optimality cuts, out of ", length(rows), " rows:")
foreach(println, cuts)

9 optimality cuts, out of 11 rows:
15 x[1] + 10 x[2] + 23 x[3] + θ[1] ≥ 385
0.44999999999999996 x[1] + 0.3 x[2] + 8.45 x[3] + θ[1] ≥ 285.4
5.3 x[1] + 0.3 x[2] + 13.3 x[3] + θ[1] ≥ 318.6
8.55 x[1] + 5.7 x[2] + 15.59 x[3] + θ[1] ≥ 351.22
5.05 x[1] + 2.1999999999999997 x[2] + 12.09 x[3] + θ[1] ≥ 324.41999999999996
3.15 x[1] + 0.3 x[2] + 11.15 x[3] + θ[1] ≥ 308.29999999999995
4.699999999999999 x[1] + 0.3 x[2] + 11.739999999999998 x[3] + θ[1] ≥ 313.91999999999996
3.9 x[1] + 0.3 x[2] + 10.94 x[3] + θ[1] ≥ 309.91999999999996
5.8 x[1] + 2.1999999999999997 x[2] + 12.84 x[3] + θ[1] ≥ 328.91999999999996


## 3. Feasibility cuts

Without the minimum capacity of 12, the first master problem buys no capacity at all — $\theta$
is not bounded yet, so nothing tells it to — and the demand cannot be met: the recourse problem is
infeasible. The module then solves an **elastic** version of the recourse problem, where artificial
variables absorb the violation of each row, and builds a feasibility cut from its multipliers.
Nothing has to be provided for that: the elastic model is derived from the data.

In [12]:
pb0 = icecream(min_capacity = 0.0)
res = lshaped(pb0; optimizer = HiGHS.Optimizer);
println("\nextensive form: ", extensive_form(pb0; optimizer = HiGHS.Optimizer)[3])

Single-cut L-shaped method
  master: HiGHS | recourse: HiGHS | scenarios: 3

 iter      lower bound   upper bound          gap              θ           Q(x)
    1   feasibility cut(s) on scenario(s) 1, 2, 3
    2             -Inf   457.000000          Inf           -Inf     385.000000
    3       325.000000   400.000000           75     205.000000     280.000000
    4       364.587629   397.787629         33.2     282.319588     315.519588
    5       374.828179   388.499725        13.67     254.828179     268.499725
    6       377.369054   383.134006        5.765     257.369054     263.134006
    7       380.281437   383.134006        2.853     260.281437     264.426347
    8       380.594325   382.499270        1.905     260.594325     262.499270
    9       381.463500   382.060820       0.5973     261.463500     262.060820
   10       381.588596   382.060820       0.4722     261.588596     262.338596
   11       381.853333   381.853333   -2.274e-13     261.853333     261.853333
con

## 4. Equality rows

A newsvendor buys $x \leq 100$ newspapers at 5 each, then sells $y_1$ of them at 12, returns $y_2$
at 2, and leaves $y_3$ of the demand $\xi$ unmet. Both rows of the second stage are equalities:

$$
y_1 + y_2 = x, \qquad y_1 + y_3 = \xi.
$$

The multiplier of an equality row is free, and the module needs no special treatment for it in the
optimality cuts. (For the feasibility cuts, the elastic model relaxes an equality by two
inequalities read in the same direction, so that its multiplier can take either sign.)

In [13]:
newsvendor = TwoStageProblem(
    c = [5.0], A = ones(1, 1), senses1 = ['<'], b = [100.0],
    q = [-12.0, -2.0, 0.0],                          # sold, returned, unmet
    W = [1.0 1.0 0.0; 1.0 0.0 1.0], senses2 = ['=', '='],
    T = reshape([-1.0, 0.0], 2, 1), h = ξ -> [0.0, ξ],
    ξ = [10.0, 20.0, 30.0, 40.0], p = [0.15, 0.35, 0.35, 0.15])

res = lshaped(newsvendor; optimizer = HiGHS.Optimizer, verbose = false)
@printf("buy %.1f newspapers, expected cost %.4f (extensive form: %.4f)\n",
        res.x[1], res.objective, extensive_form(newsvendor; optimizer = HiGHS.Optimizer)[3])

buy 30.0 newspapers, expected cost -145.0000 (extensive form: -145.0000)


## 5. Bounded recourse variables

The slides build the optimality cut as $E^Tx + \theta \geq e$ with $e = \sum_s p_s h(\xi_s)^T\pi_s$.
That formula assumes the recourse variables are only bounded below by zero: with an upper bound,
$Q(x,\xi)$ also depends on the multipliers of the bounds, and $h^T\pi$ misses them.

The module therefore builds each cut as a **supporting hyperplane** at the current iterate: the
subgradient $E$, plus the value of the subproblem, so that the cut is tight at $x^k$:
$e = \mathcal{Q}(x^k) + E^Tx^k$. With the bound $y \leq 3$:

In [14]:
bounded = icecream(ub = 3.0)
optimal = extensive_form(bounded; optimizer = HiGHS.Optimizer)[3]
tight = lshaped(bounded; optimizer = HiGHS.Optimizer, verbose = false)
@printf("supporting hyperplanes: %.6f, converged = %s (extensive form %.6f)\n",
        tight.objective, tight.converged, optimal)

supporting hyperplanes: 382.200000, converged = true (extensive form 382.200000)


`intercept = :textbook` uses the formula of the slides instead, and is kept for comparison only.
Here the lower bound crosses the upper bound at the last iteration — a warning says so — and the
method stops at a point that is not optimal, while reporting convergence:

In [15]:
textbook = lshaped(icecream(ub = 3.0, intercept = :textbook); optimizer = HiGHS.Optimizer);
@printf("closed form: %.6f, converged = %s, %.4f above the optimum\n",
        textbook.objective, textbook.converged, textbook.objective - optimal)

Single-cut L-shaped method
  master: HiGHS | recourse: HiGHS | scenarios: 3

 iter      lower bound   upper bound          gap              θ           Q(x)
    1   feasibility cut(s) on scenario(s) 2, 3
    2   feasibility cut(s) on scenario(s) 3
    3   feasibility cut(s) on scenario(s) 3
    4             -Inf   386.950000          Inf           -Inf     269.950000
    5       390.940000   384.970000        -5.97     270.940000     264.970000


┌ Warning: the lower bound exceeds the upper bound (390.93999999999994 > 384.97): the cuts are invalid, which `intercept = :textbook` causes on bounded recourse variables
└ @ Main.LShaped ~/Git/Optimization/SP_Introduction/code/LShaped.jl:1165


converged in 5 iteration(s): 1 optimality cut(s), 4 feasibility cut(s)
closed form: 384.970000, converged = true, 2.7700 above the optimum


## 6. Choosing the solvers

The master problem is a sequence of growing LPs (or MIPs, see below); the recourse problems are
small and re-solved at every iterate. `master_optimizer` and `recourse_optimizer` may therefore
differ, and each accepts anything JuMP accepts — including an `OptimizerWithAttributes`, which is
how a solver is tuned:

In [16]:
glpk_presolve = optimizer_with_attributes(GLPK.Optimizer, "presolve" => true)
res = lshaped(pb; master_optimizer = glpk_presolve, recourse_optimizer = HiGHS.Optimizer);

Single-cut L-shaped method
  master: GLPK | recourse: HiGHS | scenarios: 3

 iter      lower bound   upper bound          gap              θ           Q(x)
    1             -Inf   457.000000          Inf           -Inf     385.000000
    2       325.000000   400.000000           75     205.000000     280.000000
    3       364.587629   397.787629         33.2     282.319588     315.519588
    4       374.828179   388.499725        13.67     254.828179     268.499725
    5       377.369054   383.134006        5.765     257.369054     263.134006
    6       380.281437   383.134006        2.853     260.281437     264.426347
    7       380.594325   382.499270        1.905     260.594325     262.499270
    8       381.463500   382.060820       0.5973     261.463500     262.060820
    9       381.588596   382.060820       0.4722     261.588596     262.338596
   10       381.853333   381.853333    1.137e-13     261.853333     261.853333
converged in 10 iteration(s): 9 optimality cut(s), 0 f

## 7. Integer first-stage variables

`integer1` asks for some (or all) of the first-stage variables to be integer. The master problem
becomes a mixed-integer program; the recourse problems stay LPs, so the cuts are unchanged.

The example is a small farm. Land is bought for each of three crops, at most 500 acres in all, at
100 per acre. Once the yields are known, each crop is planted on at most the land bought for it,
its harvest must cover a contracted demand, and everything is sold: the second-stage cost of an
acre planted is its planting cost minus the sale of its harvest. In this first version the yields
are fixed to their average and the demand is random.

In [17]:
const CROPS = 3
const LAND = 500.0
const ACRE = [100.0, 100.0, 100.0]           # per acre bought
const PLANTING = [150.0, 230.0, 260.0]       # per acre planted
const PRICE = [170.0, 150.0, 36.0]           # per ton sold
const CROP_DEMAND = [200.0, 250.0, 260.0]    # tons
const AVERAGE_YIELD = [2.5, 3.0, 20.0]       # tons per acre

farm(; integer = false) = TwoStageProblem(
    c = ACRE, A = ones(1, CROPS), senses1 = ['<'], b = [LAND],
    q = PLANTING .- PRICE .* AVERAGE_YIELD,
    W = vcat(Matrix(1.0I, CROPS, CROPS), diagm(AVERAGE_YIELD)),
    senses2 = vcat(fill('<', CROPS), fill('>', CROPS)),          # yⱼ ≤ xⱼ, harvest ≥ demand
    T = vcat(-Matrix(1.0I, CROPS, CROPS), zeros(CROPS, CROPS)),
    h = ξ -> vcat(zeros(CROPS), CROP_DEMAND .* ξ),
    ξ = [0.8, 1.0, 1.25], p = fill(1 / 3, 3), integer1 = integer)

for integer in (false, true)
    res = lshaped(farm(; integer); optimizer = HiGHS.Optimizer, verbose = false)
    @printf("integer = %-5s  x = %-28s cost %.2f  (%d feasibility cuts)\n", integer,
            string(round.(res.x, digits = 3)), res.objective, res.feasibility_cuts)
end

integer = false  x = [100.0, 104.167, 295.833]    cost -136500.00  (15 feasibility cuts)
integer = true   x = [100.0, 105.0, 295.0]        cost -136300.00  (14 feasibility cuts)


In the worst case the demand for corn is $1.25 \times 250 = 312.5$ tons, which takes
$104\tfrac{1}{6}$ acres at 3 tons per acre: whole acres force 105.

## 8. Your own JuMP models

If the **yields** are random, they multiply the recourse variables, and $W$ — as well as $q$,
which contains the sales — depends on the scenario. `TwoStageProblem` accepts `W` and `q` as
functions of $\xi$, like `T` and `h`; but a model is often easier to write, and to read, as a JuMP
model than as matrices. `JuMPTwoStageProblem` accepts any JuMP models, through a few callbacks:

* `master_builder(optimizer)` returns the first-stage model and its variables `x`;
* `recourse_builder(optimizer, s)` returns the recourse model of scenario `s`, its variables, and
  the constraints whose right-hand side depends on `x`, in a fixed order;
* `rhs(x, s)` returns those right-hand sides;
* `cut_coefficients(s, x, π)` returns $T(\xi_s)^T\pi$ from the multipliers $\pi$ of those
  constraints;
* `elastic_builder(optimizer, s)`, optional, returns the elastic version of the recourse model, for
  the feasibility cuts.

The recourse models are built once per scenario; only their right-hand sides are updated between
iterations. They must be **minimized**: the module refuses a `Max` objective, whose multipliers
would give wrong cuts. Integer recourse variables are accepted, and call for the integer L-shaped
method of section 13, which then needs a lower bound on the recourse function, `recourse_bound`.

In [18]:
const YIELD = [2.0 2.4 16.0; 2.5 3.0 20.0; 3.0 3.6 24.0]    # below average, average, above

# the recourse rows of scenario s; w, if given, absorbs their violation
function farm_rows!(m, y, s; w = nothing)
    slack(k) = w === nothing ? 0.0 : w[k]
    land = [@constraint(m, y[j] - slack(j) <= 0) for j in 1:CROPS]      # yⱼ ≤ xⱼ
    for j in 1:CROPS
        @constraint(m, YIELD[s, j] * y[j] + slack(CROPS + j) >= CROP_DEMAND[j])
    end
    return land
end

farm_jump = JuMPTwoStageProblem(
    n_scenarios = 3,
    probabilities = s -> 1 / 3,
    master_builder = function (optimizer)
        m = new_model(optimizer)
        @variable(m, x[1:CROPS] >= 0)
        @constraint(m, sum(x) <= LAND)
        @objective(m, Min, dot(ACRE, x))
        return m, x
    end,
    recourse_builder = function (optimizer, s)
        m = new_model(optimizer)
        @variable(m, y[1:CROPS] >= 0)
        land = farm_rows!(m, y, s)
        @objective(m, Min, dot(PLANTING .- PRICE .* YIELD[s, :], y))
        return m, y, land
    end,
    rhs = (x, s) -> x,                         # the right-hand side of yⱼ ≤ xⱼ
    cut_coefficients = (s, x, π) -> -π,        # T = -I on these rows
    elastic_builder = function (optimizer, s)
        m = new_model(optimizer)
        @variable(m, y[1:CROPS] >= 0)
        @variable(m, w[1:2CROPS] >= 0)
        land = farm_rows!(m, y, s; w = w)
        @objective(m, Min, sum(w))
        return m, land, 1:CROPS, ones(CROPS)  # each constraint stands for its own row, sign +1
    end)

res = lshaped(farm_jump; master_optimizer = HiGHS.Optimizer, recourse_optimizer = GLPK.Optimizer)
println("x = ", round.(res.x, digits = 4))

Single-cut L-shaped method
  master: HiGHS | recourse: GLPK | scenarios: 3

 iter      lower bound   upper bound          gap              θ           Q(x)
    1   feasibility cut(s) on scenario(s) 1, 2, 3
    2   feasibility cut(s) on scenario(s) 1, 2, 3
    3   feasibility cut(s) on scenario(s) 1, 2, 3
    4             -Inf -35850.000000          Inf           -Inf  -57891.666667
    5   feasibility cut(s) on scenario(s) 1, 2, 3
    6   feasibility cut(s) on scenario(s) 1, 2, 3
    7   -136500.000000 -136500.000000    -2.91e-11 -186500.000000 -186500.000000
converged in 7 iteration(s): 1 optimality cut(s), 15 feasibility cut(s)
x = [100.0, 104.1667, 295.8333]


The cut intercept defaults to the supporting hyperplane $\mathcal{Q}(x^k) + E^Tx^k$, so that the
rows whose right-hand side does not depend on $x$ — here the demand rows — need not be listed:
their multipliers never enter the cuts.

`extensive_form` only exists for `TwoStageProblem`; the check is written out by hand:

In [19]:
m = new_model(HiGHS.Optimizer)
@variable(m, x[1:CROPS] >= 0)
@constraint(m, sum(x) <= LAND)
y = [@variable(m, [1:CROPS], lower_bound = 0) for _ in 1:3]
for s in 1:3
    land = farm_rows!(m, y[s], s)
    set_normalized_coefficient.(land, x, -1.0)          # yⱼ - xⱼ ≤ 0
end
@objective(m, Min, dot(ACRE, x) + sum(dot(PLANTING .- PRICE .* YIELD[s, :], y[s]) for s in 1:3) / 3)
optimize!(m)
@printf("extensive form: %.4f at x = %s\n", objective_value(m), round.(value.(x), digits = 4))

extensive form: -136500.0000 at x = [100.0, 104.1667, 295.8333]


## 9. Sampled scenarios

When $\xi$ follows a continuous law, the problem is approximated by a sample — the sample average
approximation of deck 09. `sample_scenarios(sampler, n; rng)` returns the pair `(ξ = ..., p = ...)`
`TwoStageProblem` takes, and can be splatted into its keyword arguments. The law of $\xi$ stays in
the hands of the modeller, as anything `rand` accepts.

`substream(seed)` is the random stream to draw from: one per experiment rather than a global
`Random.seed!`, so that a sample is reproducible on its own.

Two plants: plant 1 costs 2 per unit of capacity and nothing to run, plant 2 costs 0.5 per unit
of capacity and 3 per unit produced. The demand $\xi \sim \mathcal{N}(5, 1.5^2)$ is met by
production $y_1 \leq x_1$, $y_2 \leq x_2$, or left unmet, $y_3$, at 10 per unit.

This instance has a closed-form solution, which makes the quality of a sample visible. Plant 1 is
used first. Raising the total capacity $K = x_1 + x_2$ by one unit of plant 2 costs 0.5 and saves
$10 - 3 = 7$ whenever $\xi > K$; moving one unit of capacity from plant 2 to plant 1 costs 1.5 and
saves 3 whenever $\xi > x_1$. At the optimum,
$$
P(\xi > x_1) = \tfrac{1.5}{3} = \tfrac12, \qquad P(\xi > x_1 + x_2) = \tfrac{0.5}{7},
$$
that is $x_1^* = 5$, the median, and $x_1^* + x_2^*$ the quantile of order $13/14$.

In [20]:
two_plants(n, seed) = TwoStageProblem(;
    c = [2.0, 0.5], A = ones(1, 2), senses1 = ['<'], b = [30.0],
    q = [0.0, 3.0, 10.0],                            # plant 1, plant 2, unmet
    W = [1.0 0.0 0.0; 0.0 1.0 0.0; 1.0 1.0 1.0], senses2 = ['<', '<', '>'],
    T = [-1.0 0.0; 0.0 -1.0; 0.0 0.0], h = ξ -> [0.0, 0.0, ξ],
    sample_scenarios(Normal(5.0, 1.5), n; rng = substream(seed))...)

println("same seed, same sample:       ", two_plants(100, 1).ξ == two_plants(100, 1).ξ)
println("another seed, another sample: ", two_plants(100, 1).ξ != two_plants(100, 2).ξ)

law = Normal(5.0, 1.5)
x_star = [quantile(law, 1 / 2), quantile(law, 13 / 14) - quantile(law, 1 / 2)]
println("exact solution: x* = ", round.(x_star, digits = 4))

same seed, same sample:       true
another seed, another sample: true
exact solution: x* = [5.0, 2.1979]


Solving the approximation for growing sample sizes, with three seeds each, shows the solution
settling on $x^*$, and the spread between seeds shrinking. The multicut version shines here: one
cut per scenario describes $\mathcal{Q}$ much faster than their average does.

In [21]:
println("     n   seed   objective        x₁        x₂   iterations (multi / single)")
for n in (10, 100, 1000), seed in 1:3
    pbn = two_plants(n, seed)
    res = lshaped(pbn; optimizer = HiGHS.Optimizer, cuts = :multi, verbose = false)
    res1 = lshaped(pbn; optimizer = HiGHS.Optimizer, verbose = false)
    @printf("%6d %6d %11.4f %9.4f %9.4f %8d / %d\n", n, seed, res.objective, res.x...,
            res.iterations, res1.iterations)
end

     n   seed   objective        x₁        x₂   iterations (multi / single)
    10      1     13.0038    4.7590    2.8258        4 / 11
    10      2     11.4387    5.1856    0.9867        4 / 13
    10      3     11.8198    4.6564    2.5103        4 / 10
   100      1     12.6623    4.8282    1.9950        4 / 21
   100      2     13.3998    5.0623    2.2762        4 / 17
   100      3     13.2857    4.9436    2.2230        4 / 18
  1000      1     13.1958    5.0068    2.1476        4 / 27
  1000      2     13.2334    4.9135    2.3472        4 / 26
  1000      3     13.4985    5.1214    2.2030        4 / 26


The sample is part of the problem: the single-cut and the multicut runs above see exactly the same
scenarios, because both are handed the same `TwoStageProblem`, built once from its own stream.

## 10. The value of perfect information and of the stochastic solution

Deck 03 compares the optimal value $\mathrm{RP}$ of the recourse problem with two simpler values:

* the **wait-and-see** value $\mathrm{WS} = \mathbb{E}_\xi[\min_x z(x,\xi)]$, if $\xi$ were known
  before deciding $x$, which gives the **expected value of perfect information**
  $\mathrm{EVPI} = \mathrm{RP} - \mathrm{WS}$;
* the **expected result** $\mathrm{EEV}$ of the decision $\overline{x}(\overline{\xi})$ obtained by
  replacing $\xi$ by its mean, which gives the **value of the stochastic solution**
  $\mathrm{VSS} = \mathrm{EEV} - \mathrm{RP}$.

The module computes each of them. We take the farmer of Birge and Louveaux (Section 1.1): 500
acres of wheat, corn and sugar beets, with random yields; wheat and corn are bought ($w$) or sold
($y_1, y_2$) to feed the cattle, beets sold at 36 up to a quota of 6000 tons ($y_3$) and at 10
beyond it ($y_4$).

In [22]:
farmer = TwoStageProblem(
    c = [150.0, 230.0, 260.0], A = ones(1, 3), senses1 = ['<'], b = [500.0],
    q = [238.0, 210.0, -170.0, -150.0, -36.0, -10.0],          # w₁, w₂, y₁, y₂, y₃, y₄
    W = [1.0 0.0 -1.0 0.0 0.0 0.0;                              # t₁x₁ + w₁ - y₁ ≥ 200
         0.0 1.0 0.0 -1.0 0.0 0.0;                              # t₂x₂ + w₂ - y₂ ≥ 240
         0.0 0.0 0.0 0.0 1.0 1.0;                               # y₃ + y₄ ≤ t₃x₃
         0.0 0.0 0.0 0.0 1.0 0.0],                              # y₃ ≤ 6000
    senses2 = ['>', '>', '<', '<'],
    T = t -> [t[1] 0.0 0.0; 0.0 t[2] 0.0; 0.0 0.0 -t[3]; 0.0 0.0 0.0],
    h = [200.0, 240.0, 0.0, 6000.0],
    ξ = [[3.0, 3.6, 24.0], [2.5, 3.0, 20.0], [2.0, 2.4, 16.0]], p = fill(1 / 3, 3))

rp = lshaped(farmer; optimizer = HiGHS.Optimizer, verbose = false).objective
ws = wait_and_see(farmer; optimizer = HiGHS.Optimizer)
ev = expected_value_problem(farmer; optimizer = HiGHS.Optimizer)
eev = expected_result(farmer, ev.x; optimizer = HiGHS.Optimizer)
@printf("WS  = %12.2f   decisions per scenario: %s\n", ws.value,
        join(("(" * join(round.(d, digits = 2), ", ") * ")" for d in ws.decisions), ", "))
@printf("RP  = %12.2f\n", rp)
@printf("EV  = %12.2f   at x̄ = %s\n", ev.value, round.(Int, ev.x))
@printf("EEV = %12.2f\n", eev)

WS  =   -115405.56   decisions per scenario: (183.33, 66.67, 250.0), (120.0, 80.0, 300.0), (100.0, 25.0, 375.0)
RP  =   -108390.00
EV  =   -118600.00   at x̄ = [120, 80, 300]
EEV =   -107240.00


The values of Birge and Louveaux: $\mathrm{EV} \leq \mathrm{WS} \leq \mathrm{RP} \leq \mathrm{EEV}$
(here, with costs, $-118600 \leq -115406 \leq -108390 \leq -107240$). `evpi` and `vss` compute the
two differences directly; `RP` is obtained by the L-shaped method, unless passed as `rp`:

In [23]:
@printf("EVPI = %.2f\n", evpi(farmer; rp = rp, optimizer = HiGHS.Optimizer))
@printf("VSS  = %.2f\n", vss(farmer; rp = rp, optimizer = HiGHS.Optimizer))

EVPI = 7015.56
VSS  = 1150.00


$\mathrm{EEV}$, and therefore $\mathrm{VSS}$, can be infinite. Without its minimum capacity, the
ice-cream problem of section 3 buys, for the mean demand, too little capacity for the largest one:

In [24]:
x̄ = expected_value_problem(pb0; optimizer = HiGHS.Optimizer).x
println("capacity bought for the mean demand: ", round(sum(x̄), digits = 4))
println("EEV = ", expected_result(pb0, x̄; optimizer = HiGHS.Optimizer), ", VSS = ", vss(pb0; optimizer = HiGHS.Optimizer))

capacity bought for the mean demand: 10.0
EEV = Inf, VSS = Inf


`wait_and_see`, `expected_result`, `evpi`, and `vss` with a given $\overline{x}$, work as well on
JuMP models of your own: `vss(model, x̄; ...)`. Only `expected_value_problem` needs the scenarios
of a `TwoStageProblem`, to average them.

## 11. The variants of deck 04

### Clusters of scenarios

Between one cut per iteration and one per scenario, the "hybrid approaches" of deck 04 group the
scenarios into clusters $\mathcal{S}_1, \dots, \mathcal{S}_C$, with one $\theta_k$ each: `cuts = C`
makes $C$ clusters of consecutive scenarios, and a vector of vectors gives them explicitly. Exercise 6
of Birge and Louveaux (Section 5.1) takes $Q(x, \xi) = |x - \xi|$ on $0 \leq x \leq 10$, with $\xi$
taking the values $0.5, 1, 1.5, 3, 4, 5$ with probability $1/9$ and $2$ with probability $1/3$; it
groups the scenarios around $1$, $2$ and $4$. Starting from $x^1 = 0$ (`x0`, below):

In [25]:
absdev = TwoStageProblem(
    c = [0.0], A = ones(1, 1), senses1 = ['<'], b = [10.0],
    q = [1.0, 1.0], W = [1.0 -1.0], senses2 = ['='], T = ones(1, 1), h = ξ -> [ξ],
    ξ = [0.5, 1.0, 1.5, 2.0, 3.0, 4.0, 5.0], p = [1, 1, 1, 3, 1, 1, 1] ./ 9)
for cuts in (:single, [[1, 2, 3], [4], [5, 6, 7]], :multi)
    res = lshaped(absdev; optimizer = HiGHS.Optimizer, cuts = cuts, x0 = [0.0], verbose = false)
    @printf("%-28s iterates %-28s %2d cuts, objective %.4f\n", cuts,
            join(round.([h.x[1] for h in res.history], digits = 3), ", "), res.optimality_cuts,
            res.objective)
end

single                       iterates 0.0, 10.0, 2.333, 1.5, 2.0    4 cuts, objective 1.0000
[[1, 2, 3], [4], [5, 6, 7]]  iterates 0.0, 10.0, 2.0                6 cuts, objective 1.0000
multi                        iterates 0.0, 10.0, 2.0               14 cuts, objective 1.0000


The three clusters need as few iterations as the multicut version, with half its cuts: their
conditional expectations are already piecewise linear with a single kink each.

### Regularized decomposition and trust region

The single-cut iterates jump from one end of the domain to the other. Deck 04 stabilizes them around
a centre $a^\nu$, either with a proximal term, `regularization = :regularized_decomposition`
($\min c^Tx + \sum_k P_k\theta_k + \|x - a^\nu\|^2 / (2\rho)$, `rho` = 1 by default), or with a
box $\|x - a^\nu\|_\infty \leq \Delta$, `regularization = :trust_region`. The example of the deck is
Exercise 5 of Birge and Louveaux, started at $a^1 = -0.5$: the regularized iterates reach the
optimum $x = 0.25$ at once, where the plain method, started at $-2$, visits $20$ and $12/7$.

In [26]:
ex5 = TwoStageProblem(
    c = [0.0], A = zeros(0, 1), senses1 = Char[], b = Float64[],
    q = ξ -> ξ[1], W = [1.0 -1 -1 -1 0 0; 0 1 0 0 1 0; 0 0 1 0 0 1], senses2 = ['=', '=', '='],
    T = reshape([1.0, 0, 0], 3, 1), h = ξ -> ξ[2],
    ξ = [([1.0, 0, 0, 0, 0, 0], [-1.0, 2, 7]), ([1.5, 0, 2 / 7, 1, 0, 0], [0.0, 2, 7])],
    p = [0.5, 0.5], lb1 = -20.0, ub1 = 20.0)
rd = lshaped(ex5; optimizer = HiGHS.Optimizer, cuts = :multi, x0 = [-0.5],
             regularization = :regularized_decomposition)
println()
for (label, kw) in (("plain", (;)), ("trust region", (regularization = :trust_region, radius = 1.0)))
    res = lshaped(ex5; optimizer = HiGHS.Optimizer, x0 = [-2.0], verbose = false, kw...)
    @printf("%-13s from -2: iterates %s\n", label, join(round.([h.x[1] for h in res.history], digits = 3), ", "))
end

Multi-cut L-shaped method, regularized decomposition (rho = 1.0)
  master: HiGHS | recourse: HiGHS | scenarios: 2

 iter      model value   upper bound          gap         min θₖ           Q(x)
    1             -Inf     0.375000          Inf           -Inf       0.375000
    2        -0.187500     0.000000       0.5625      -0.375000       0.000000
converged in 3 iteration(s): 3 optimality cut(s), 0 feasibility cut(s)

plain         from -2: iterates -2.0, 20.0, 1.714, -0.4, -0.0
trust region  from -2: iterates -2.0, -1.0, 1.0


The master of regularized decomposition is a quadratic program, and the choice of its solver
matters: GLPK cannot solve it, and HiGHS's QP solver often returns wrong solutions, declared optimal,
even on small masters. Before stopping, `lshaped` checks that the last master was solved, and
raises an error if not; Ipopt solves them, provided it may not relax the constraints. Without
regularization, its log reports a lower bound; with it, the "model value" $m(x^\nu)$ of the master
solution, which the method stops on once it reaches the value $f(a^\nu)$ of the centre. Up to a
tolerance, this test is weak — the proximal term flattens the model — so regularized decomposition
also solves the master without it, an LP whose value is a lower bound, and only stops if this bound
meets the incumbent; otherwise, the solution of this LP is the next iterate.

In [27]:
qp = optimizer_with_attributes(Ipopt.Optimizer, "bound_relax_factor" => 0.0, "sb" => "yes")
res = lshaped(farmer; master_optimizer = qp, recourse_optimizer = HiGHS.Optimizer,
              regularization = :regularized_decomposition, rho = 100.0)
@printf("objective %.2f, lower bound %.2f\n", res.objective, res.lower_bound)

Single-cut L-shaped method, regularized decomposition (rho = 100.0)
  master: IpoptMathOptInterfaceExt | recourse: HiGHS | scenarios: 3

 iter      model value   upper bound          gap              θ           Q(x)
    1             -Inf 98000.000000          Inf           -Inf   98000.000000
    2   -132000.000000 -28000.000000      2.3e+05 -262000.000000 -158000.000000
    3   -129000.000000 -98200.000000     1.01e+05 -237000.000000 -206200.000000
    4   -126813.880126 -98200.000000    2.861e+04 -218782.334385 -173606.309148
    5   -121963.636364 -104220.606061    2.376e+04 -238745.454545 -221002.424242
    6   -112120.979021 -107259.440559         7900 -222603.496503 -217741.958042
    7   -110419.485842 -107259.440559         3160 -220134.463487 -214455.476900
    8   -109504.204117 -107812.583357         2245 -221262.974775 -219571.354016
    9   -108861.111111 -108327.777778         1049 -218472.222222 -217938.888889
   10   -108496.153846 -108327.777778        168.4 -216888.

### Starting at the mean-value decision, and stopping early

`x0 = :mean_value` evaluates the subproblems at $\overline{x}$ before the first master: the first
iteration gives $\mathrm{EEV}$ and a first cut. Along the iterations, the bounds
$L^\nu \leq \mathrm{RP} \leq U^\nu$ give, as deck 04 shows,

$$
L^\nu - \mathrm{WS} \leq \mathrm{EVPI} \leq U^\nu - \mathrm{WS},
\qquad
\mathrm{EEV} - U^\nu \leq \mathrm{VSS} \leq \mathrm{EEV} - L^\nu,
$$

which `evpi_bounds(entry, ws)` and `vss_bounds(entry, eev)` compute for an entry of the history. A
`callback` receives each entry and stops the method by returning `true`: to know whether solving the
stochastic problem is worth it, it is enough that the lower bound on the VSS be positive.

In [28]:
eev = expected_result(farmer, expected_value_problem(farmer; optimizer = HiGHS.Optimizer).x;
                      optimizer = HiGHS.Optimizer)
function report(entry)
    lo, hi = vss_bounds(entry, eev)
    @printf("iteration %d: %10.2f ≤ VSS ≤ %10.2f\n", entry.iteration, lo, hi)
    return lo > 0
end
early = lshaped(farmer; optimizer = HiGHS.Optimizer, x0 = :mean_value, verbose = false, callback = report)
println("stopped: ", early.stopped, " after ", early.iterations, " iterations; incumbent ", round.(early.x, digits = 1))

iteration 1:       0.00 ≤ VSS ≤        Inf
iteration 2:       0.00 ≤ VSS ≤   81160.00
iteration 3:       0.00 ≤ VSS ≤   33292.49
iteration 4:       0.00 ≤ VSS ≤    9210.29
iteration 5:       0.00 ≤ VSS ≤    3158.35
iteration 6:    1150.00 ≤ VSS ≤    2110.00
stopped: true after 6 iterations; incumbent [170.0, 80.0, 250.0]


## 12. Instances in the SMPS format

Published test problems of stochastic programming come in the **SMPS** format: a *core* file, the
deterministic LP in MPS format; a *time* file, which says where the second stage starts; a *stoch*
file, the law of the random data. `read_smps` reads them into an `SMPSProblem`: the data of both
stages, the random elements — right-hand sides, coefficients of $T$ or $W$, costs $q$ — and their
law, a sampler of the vectors $\xi$ of their values. `enumerate_scenarios(smps.law)` lists its
support when it is small, `sample_scenarios(smps.law, n)` samples it otherwise; either goes to
`TwoStageProblem(smps; ξ, p)`.

A small instance, written out here: two plants whose capacity is bought, a random demand (`INDEP`:
each element on its own), and prices that move together (`BLOCKS`: a block of elements whose values
are drawn jointly), one of them a coefficient of $W$.

In [29]:
dir = mktempdir()
write(joinpath(dir, "toy.cor"), """
NAME          TOY
ROWS
 N  COST
 L  BUDGET
 G  DEMAND
 L  CAP1
 E  BAL
COLUMNS
    X1        COST         2.0   BUDGET       1.0
    X1        CAP1        -1.0
    X2        COST         3.0   BUDGET       1.0
    X2        BAL         -1.0
    Y1        COST         4.0   DEMAND       1.0
    Y1        CAP1         1.0
    Y2        COST         6.0   DEMAND       1.0
    Y2        BAL          1.0
    Y3        COST        10.0   DEMAND       1.0
RHS
    RHS       BUDGET      10.0   DEMAND       5.0
BOUNDS
 UP BND       Y3          20.0
ENDATA
""")
write(joinpath(dir, "toy.tim"), """
TIME          TOY
PERIODS       LP
    X1        BUDGET                   STAGE1
    Y1        DEMAND                   STAGE2
ENDATA
""")
write(joinpath(dir, "toy.sto"), """
STOCH         TOY
INDEP         DISCRETE
    RHS       DEMAND       4.0   STAGE2   0.3
    RHS       DEMAND       7.0   STAGE2   0.7
BLOCKS        DISCRETE
 BL PRICES    STAGE2       0.5
    Y3        COST        10.0
 BL PRICES    STAGE2       0.5
    Y3        COST        14.0
    Y1        CAP1         2.0
ENDATA
""")
toy = read_smps(joinpath(dir, "toy"))
println(toy)
println("random elements: ", toy.elements, " named ", toy.names)
ξ, p = enumerate_scenarios(toy.law)
for (point, w) in zip(ξ, p)
    println("  ξ = ", point, "  with probability ", w)
end
pb_toy = TwoStageProblem(toy; ξ = ξ, p = p)
res = lshaped(pb_toy; optimizer = HiGHS.Optimizer, cuts = :multi, verbose = false)
@printf("L-shaped %.4f, extensive form %.4f\n", res.objective,
        extensive_form(pb_toy; optimizer = HiGHS.Optimizer)[3])

SMPS instance TOY: 2 first-stage variables and 1 rows, 3 second-stage variables and 3 rows, 3 random elements in 2 independent blocks, 4 scenarios
random elements: [(:h, 1, 0), (:q, 3, 0), (:W, 2, 1)] named [("RHS", "DEMAND"), ("Y3", "COST"), ("Y1", "CAP1")]
  ξ = [4.0, 10.0, 1.0]  with probability 0.15
  ξ = [4.0, 14.0, 2.0]  with probability 0.15
  ξ = [7.0, 10.0, 1.0]  with probability 0.35
  ξ = [7.0, 14.0, 2.0]  with probability 0.35
L-shaped 50.9000, extensive form 50.9000


The test problems of Linderoth, Shapiro and Wright (2006) have far too many scenarios to be
enumerated — $10^6$ for LandS, $10^{70}$ for SSN — and are solved on samples: the sample average
approximation. `LShaped_instances.jl` downloads them on first use. On LandS, the value of the
approximation, and that of its solution on an independent sample, against the optimal value they
estimate, 225.62:

In [30]:
include("LShaped_instances.jl")
lands = lsw_instance("LandS")
println(lands)
for n in (50, 200, 1000)
    saa = TwoStageProblem(lands; sample_scenarios(lands.law, n; rng = substream(1))...)
    res = lshaped(saa; optimizer = HiGHS.Optimizer, cuts = :multi, verbose = false)
    test = TwoStageProblem(lands; sample_scenarios(lands.law, 5000; rng = substream(2))...)
    @printf("n = %4d: approximation %8.3f, its solution on 5000 new scenarios %8.3f\n", n,
            res.objective, expected_result(test, res.x; optimizer = HiGHS.Optimizer))
end

SMPS instance LandS: 4 first-stage variables and 2 rows, 12 second-stage variables and 7 rows, 3 random elements in 3 independent blocks, 1000000 scenarios
n =   50: approximation  226.001, its solution on 5000 new scenarios  224.827
n =  200: approximation  227.170, its solution on 5000 new scenarios  224.745
n = 1000: approximation  225.083, its solution on 5000 new scenarios  224.726


In expectation, the value of the approximation lies below the optimum and the true cost of its
solution above. The cost on 5000 new scenarios is itself an estimate: with a standard deviation of
the cost of about 58 here, its error is about ±1.6 (two standard errors), which covers the gap to
225.62. The decisions themselves are already good at $n = 50$: LandS is a small, well-behaved
problem.

## 13. Integer recourse

With integer second-stage variables (`integer2`), $\mathcal{Q}(\cdot, \xi)$ is no longer convex,
and the multipliers of the recourse problems give no cut. On **binary** first-stage variables — those
the second stage depends on — the integer L-shaped method of Laporte and Louveaux (Birge and
Louveaux, 2011, Section 7.2) still works: at a binary iterate $x^\nu$, with $S$ its set of ones and
$L$ a lower bound on $\mathcal{Q}$,

$$
\theta \geq (q_S - L)\Big(\sum_{i \in S} x_i - \sum_{i \notin S} x_i - |S| + 1\Big) + L,
\qquad q_S = \mathcal{Q}(x^\nu),
$$

is exact at $x^\nu$ and below $L$ at every other binary point. `lshaped` adds these cuts, and the
cuts of the LP relaxation of the recourse problems, which remain valid since they bound
$\mathcal{Q}$ from below; the master is a mixed-integer program, solved anew at each iteration by
the solver, which does the branching. $L$ is computed from the LP relaxation of both stages, scenario
by scenario, unless given as `recourse_bound`. Example 1 of the book, at $x = (0, 1)$:
$L = -5.75$, $q_S = -5$, and the cut $\theta \geq 0.75(x_2 - x_1) - 5.75$.

In [31]:
integer_example(c) = TwoStageProblem(
    c = c, A = zeros(0, 2), senses1 = Char[], b = Float64[],
    q = [-2.0, -3.0], W = [1.0 2.0; 1.0 0.0], senses2 = ['<', '<'], T = [1.0 0.0; 0.0 1.0],
    h = ξ -> ξ, ξ = [[2.0, 2.0], [4.0, 3.0]], p = [0.5, 0.5], ub1 = 1.0, integer1 = true,
    integer2 = true)
res = lshaped(integer_example([-1.0, -1.0]); optimizer = HiGHS.Optimizer, x0 = [0.0, 1.0])
for con in all_constraints(res.master.model; include_variable_in_set_constraints = false)
    println("  ", con)
end

Single-cut integer L-shaped method
  master: HiGHS | recourse: HiGHS | scenarios: 2

 iter      lower bound   upper bound          gap              θ           Q(x)
    1             -Inf    -6.000000          Inf           -Inf      -5.000000
    2        -6.000000    -6.000000            0      -5.000000      -5.000000
converged in 2 iteration(s): 1 optimality cut(s), 1 integer optimality cut(s), 0 feasibility cut(s)
  -1.5 x[1] - 0.5 x[2] + θ[1] ≥ -5.75
  0.75 x[1] - 0.75 x[2] + θ[1] ≥ -5.75


The first cut is that of the LP relaxation, the second the integer cut of the book. A larger
instance: servers to open (binary), clients that show up at random and are each assigned to one open
server (binary), capacities, and an overflow at a penalty — after the SSLP instances of Ntaimo and
Sen (2005).

In [32]:
function server_location(J, I, S; seed = 1)
    rng = substream(seed)
    f = rand(rng, 40:80, J) .* 1.0               # opening costs
    r = rand(rng, 5:25, I, J) .* 1.0             # revenues
    d = rand(rng, 5:25, I) .* 1.0                # demands
    yi(i, j) = (i - 1) * J + j
    W, T = zeros(J + I, I * J + J), zeros(J + I, J)
    for j in 1:J
        for i in 1:I
            W[j, yi(i, j)] = d[i]                # Σᵢ dᵢ yᵢⱼ - zⱼ ≤ u xⱼ
            W[J + i, yi(i, j)] = 1.0             # Σⱼ yᵢⱼ = ξᵢ, client i present or not
        end
        W[j, I * J + j] = -1.0
        T[j, j] = -sum(d) / 2
    end
    return TwoStageProblem(c = f, A = zeros(0, J), senses1 = Char[], b = Float64[],
        q = vcat([-r[i, j] for i in 1:I for j in 1:J], fill(1000.0, J)), W = W,
        senses2 = vcat(fill('<', J), fill('=', I)), T = T, h = ξ -> vcat(zeros(J), ξ),
        ξ = [Float64.(rand(rng, I) .< 0.5) for _ in 1:S], p = fill(1 / S, S),
        ub1 = 1.0, integer1 = true, ub = vcat(ones(I * J), fill(Inf, J)),
        integer2 = vcat(trues(I * J), falses(J)))
end
servers = server_location(5, 25, 50)
for cuts in (:single, :multi)
    t = @elapsed res = lshaped(servers; optimizer = HiGHS.Optimizer, cuts = cuts, verbose = false)
    @printf("%-6s %10.4f  open: %s  %3d iterations, %4d LP cuts, %4d integer cuts, %.2f s\n", cuts,
            res.objective, findall(res.x .> 0.5), res.iterations, res.optimality_cuts,
            res.integer_cuts, t)
end
t = @elapsed (_, x, value) = extensive_form(servers; optimizer = HiGHS.Optimizer)
@printf("extensive form %10.4f  open: %s  %.2f s\n", value, findall(x .> 0.5), t)

single  -123.9600  open: [3, 5]   14 iterations,   13 LP cuts,   13 integer cuts, 3.50 s
multi   -123.9600  open: [3, 5]   13 iterations,  599 LP cuts,  599 integer cuts, 13.19 s
extensive form  -123.9600  open: [3, 5]  8.50 s
